# 🔬 Notebook 01: Exploratory Data Analysis (EDA)
## MoNuSeg Quantitative Histopathology Analysis

This notebook explores the dataset properties including:
1. **Image Properties**: Dimensions, tissue area fraction, RGB & LAB color distributions.
2. **Stain Deconvolution**: Optical density transformation & Macenko stain matrix estimation.
3. **Ground Truth Annotations**: XML polygon extraction, binary & instance masks.
4. **Organ-Level Staining & Density Variation**: Nuclei count, density, and area distributions across organ groups.

In [ ]:
import os
import sys
sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

from preprocessing import create_tissue_mask, macenko_normalize, load_image, xml_to_masks
from features import extract_dataset_morphology, nuclei_density
from visualization import plot_image_and_mask, plot_color_distribution

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 6)
print("Modules imported successfully.")

### 1. XML Annotation Parsing & Ground Truth Rasterization
MoNuSeg provides individual polygonal annotations for each nucleus. We parse the XML to produce:
- **Binary Mask**: $0 = \text{background}, 1 = \text{nucleus}$
- **Instance Mask**: $0 = \text{background}, 1, 2, \dots, N = \text{distinct nuclei}$

In [ ]:
# Example synthetic or real XML parse test
data_dir = Path("../data/MoNuSeg")
print(f"Checking dataset path: {data_dir.resolve()}")
if data_dir.exists():
    images = list(data_dir.glob("images/*.tif")) + list(data_dir.glob("images/*.png"))
    print(f"Found {len(images)} images in dataset.")
else:
    print("MoNuSeg data directory not yet populated. (Follow setup instructions in README)")

### 2. Tissue Masking & Optical Density (Macenko) Analysis
White glass slide areas distort stain vectors. Tissue masking isolates active biological tissue prior to SVD optical density calculation.

In [ ]:
# Demonstration on a synthetic tissue patch or sample
np.random.seed(42)
# Create synthetic H&E-like patch (stain simulation)
sample_patch = np.full((512, 512, 3), 245, dtype=np.uint8) # glass background
cv2.circle(sample_patch, (256, 256), 180, (180, 100, 190), -1) # eosin/tissue background
for _ in range(40):
    cx, cy = np.random.randint(140, 370, 2)
    cv2.circle(sample_patch, (cx, cy), np.random.randint(8, 16), (70, 20, 110), -1) # hematoxylin nuclei

tissue_mask = create_tissue_mask(sample_patch)
norm_patch, stain_matrix = macenko_normalize(sample_patch, tissue_mask=tissue_mask)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(sample_patch)
axes[0].set_title("Original Sample (H&E)")
axes[0].axis("off")

axes[1].imshow(tissue_mask, cmap="gray")
axes[1].set_title(f"Tissue Mask (Fraction: {tissue_mask.mean():.1%})")
axes[1].axis("off")

axes[2].imshow(norm_patch)
axes[2].set_title("Macenko Normalized")
axes[2].axis("off")
plt.tight_layout()
plt.show()

### 3. Quantitative Nuclear Morphology & Density Distributions
We extract quantitative cell-level features:
- Area (pixels)
- Circularity ($4\pi A / P^2$)
- Eccentricity (major/minor axis)
- Nearest neighbor distance

In [ ]:
print("Morphological analysis workflow ready for whole dataset execution.")